# Food Trust -- Product Database Build (Epic 2)

Builds `products.db` from the Open Food Facts `food.parquet` export.

Pipeline: `food.parquet` -> filter to `en:baby-foods` -> `build/baby_foods_raw.jsonl` -> clean & flatten -> `products.db`

Rebuilt 2026-08-31 (first pass) to add 16 fields confirmed in the field review of all 111 OFF columns (see `product_data/raw_data/`), fix `image_url` selection, and normalize empty values to NULL.

Rebuilt again 2026-08-31 (second pass, this notebook) after a follow-up review of all 38 columns against real sample data, answering 24 open questions and resolving every "still open" item from the first pass. `build/baby_foods_raw.jsonl` was reused unchanged -- every change this pass is in Step 2 (cleaning/flattening), not in what's extracted from `food.parquet`.

Rebuilt again 2026-09-24 (Step 2 only) to prep for the OCR handoff: per-nutrient `_source`/`_quality_flag` columns on all 8 core nutrients (previously only `sodium_source` existed), a `missing_nutrient_count` column, and a new `product_images` table capturing every OFF photo per product instead of discarding `images` after computing `image_url`. See `OCR_HANDOFF.md` for how the OCR pass is meant to use this. Re-run against a fresh `food.parquet` pull that day found 38 newly-added and 2 dropped baby-food products since the 2026-08-31 snapshot (net +36 rows, 6,920 -> 6,956) -- ordinary upstream drift, not a logic change.

## Step 1 -- Extract baby-food products from food.parquet

In [ ]:
"""
Step 1: extract all OFF products tagged 'en:baby-foods' from food.parquet
into build/baby_foods_raw.jsonl.

Parquet over CSV: the 13GB CSV times out reading off this mount; Parquet
(7.8GB, columnar) lets us read only COLS, in row-group batches.

Batching/checkpointing: each call has a time budget (TIME_BUDGET below);
this loop processes BATCH_SIZE row groups per call, appends matches, and
checkpoints progress in checkpoint.txt. Re-run until it prints DONE.
BATCH_SIZE=1000 OOM'd previously; 200 is safe.

Filter: categories_tags contains 'en:baby-foods'. Validated: en:baby-milks
and en:infant-formulas are both subsets of en:baby-foods in OFF's
taxonomy (union == 7036, same as en:baby-foods alone); 40-row manual
spot check found zero false positives.

2026-08-31 rebuild: added 16 fields confirmed in this session's field
review (see product_data/raw_data/) -- countries_tags, last_modified_t,
product_quantity(+unit), serving_quantity, lang, obsolete, brands_tags,
generic_name, link, manufacturing_places_tags, food_groups_tags,
minerals_tags, vitamins_tags, nucleotides_tags, nutrition_data_per.

2026-09-24 note: COLS below is unchanged -- `images` was already being
extracted in full here (it was needed for image_url). Step 2 now also
uses it to build a product_images table, so no change was needed in this
cell; only Step 2's flattening logic changed.
"""
import pyarrow.parquet as pq
import json, time, sys
from pathlib import Path

BASE_DIR = Path.cwd()
PARQUET = BASE_DIR.parent / "raw_fooddata" / "food.parquet"
BUILD_DIR = BASE_DIR / "build"
BUILD_DIR.mkdir(exist_ok=True)
CHECKPOINT = BUILD_DIR / "checkpoint.txt"
OUTFILE = BUILD_DIR / "baby_foods_raw.jsonl"
BATCH_SIZE = int(sys.argv[1]) if len(sys.argv) > 1 else 300
TIME_BUDGET = 150  # seconds

COLS = [
    "code", "product_name", "generic_name", "brands", "brands_tags",
    "quantity", "product_quantity", "product_quantity_unit",
    "serving_size", "serving_quantity", "categories_tags",
    "countries_tags", "manufacturing_places_tags", "lang", "link",
    "nutriments", "nutrition_data_per", "images", "ingredients_text",
    "ingredients_tags", "additives_tags", "allergens_tags", "labels_tags",
    "food_groups_tags", "minerals_tags", "vitamins_tags",
    "nucleotides_tags", "nutriscore_grade", "nova_group",
    "last_modified_t", "obsolete",
]

def main():
    t0 = time.time()
    pf = pq.ParquetFile(str(PARQUET))
    n_rg = pf.num_row_groups
    start = int(CHECKPOINT.read_text().strip()) if CHECKPOINT.exists() else 0
    if start >= n_rg:
        print(f"DONE. All {n_rg} row groups already processed.")
        return

    total_matches = sum(1 for _ in open(OUTFILE)) if OUTFILE.exists() else 0
    out = open(OUTFILE, "a")
    rg = start
    while rg < n_rg and (time.time() - t0) < TIME_BUDGET:
        batch_end = min(rg + BATCH_SIZE, n_rg)
        tbl = pf.read_row_groups(list(range(rg, batch_end)), columns=COLS)
        cats = tbl.column("categories_tags").to_pylist()
        match_idx = [i for i, c in enumerate(cats) if c and "en:baby-foods" in c]
        if match_idx:
            for r in tbl.take(match_idx).to_pylist():
                out.write(json.dumps(r, default=str) + "\n")
            total_matches += len(match_idx)
        rg = batch_end
        CHECKPOINT.write_text(str(rg))
    out.close()

    elapsed = round(time.time() - t0, 1)
    pct = round(100 * rg / n_rg, 1)
    print(f"Processed up to row group {rg}/{n_rg} ({pct}%) in {elapsed}s. Total matches: {total_matches}.")
    if rg >= n_rg:
        print("DONE.")

if __name__ == "__main__":
    main()


## Step 2 -- Clean, flatten, and build products.db

In [ ]:
"""
Step 2: clean and flatten build/baby_foods_raw.jsonl into products.db.

Language selection (pick_text): product_name / ingredients_text /
generic_name are lists of {lang, text}. Prefer 'en', then 'main', then
first available.

name: falls back to generic_name when product_name has no usable entry
(covers ~201 rows that were previously blank).

Rows with no name AND no brand are dropped entirely (2026-08-31: 116
rows where a user could neither find nor recognize the product).

brands -> brand, brands_tags -> brand_tags (2026-08-31 rename for
clarity/consistency).

product_quantity: OFF sometimes ships a structured value of 0 with no
unit alongside garbage free-text quantity (e.g. quantity='1'). A parsed
quantity of exactly 0 is treated as "no data", not a real packet
weight, and is nulled out (unit nulled with it) (2026-08-31).

last_modified_date: ISO 8601 date derived from last_modified_t (Unix
epoch) so the frontend doesn't have to convert it itself (2026-08-31,
added for the IT feedback "last updated: Unavailable" fix).

ingredients_text: stripped of residual HTML markup (e.g.
'<span class="allergen">WHEY</span>' -> 'WHEY') left over in some OFF
records; case is left as-is (2026-08-31).

Dropped columns (2026-08-31, low value for this baby-food subset):
manufacturing_places_tags (10.1% coverage, no backfill source),
link (10.3% coverage), nucleotides_tags (2.0% coverage). Not revisited
in the 2026-09-24 pass below -- none of the three are printed on a
nutrition panel, so OCR can't backfill them either.

Sodium (sodium_mg_100g / sodium_source): OFF stores sodium in g/100g,
converted to mg (x1000). If sodium itself is missing but salt is present,
sodium is derived as salt_g / 2.5 * 1000; sodium_source records which
happened ('sodium' / 'salt_derived' / None) so a derived figure is never
presented as directly measured.

image_url: picks the image OFF's own images[].key explicitly marks as a
front photo -- product's own language first, then 'front_en', then any
other 'front_*' -- and returns None if nothing is explicitly marked
front. (Previously fell back to images[0], which could be a non-front
photo; that fallback is removed.) URL path follows OFF's own barcode
convention (3-3-3-rest digit grouping). Kept as-is for backward
compatibility with existing frontend code that reads it -- the new
product_images table below is the complete picture, this column is just
a convenience default.

obsolete: discontinued products (obsolete=true) are excluded from the
build entirely, not just flagged.

Empty/missing values: every text and list field is normalized to SQL
NULL when missing (previously some columns mixed NULL and '' or stored
'[]' for both "confirmed empty" and "no data" -- that distinction was
already lost in this data, so it's not being reconstructed; NULL is the
single "no value" representation going forward).

2026-09-24 additions (OCR handoff prep -- see OCR_HANDOFF.md):

1. Per-nutrient {nutrient}_source / {nutrient}_quality_flag columns for
   all 8 core nutrients, extending the sodium_source pattern that already
   existed. source is 'off_measured' (or 'salt_derived' for sodium) when
   this build populated the value, else NULL; quality_flag is 'confirmed'
   when populated, else NULL. Left for the OCR pass to overwrite with
   'ocr_extracted' / 'unverified' once photos are read -- no further
   schema change needed for that.

2. missing_nutrient_count (0-8) per product, computed once here instead
   of via a separate query every time someone asks "what's our missing-
   data ratio".

3. New product_images table, one row per product that has at least one
   OFF photo (PK = code, so products with zero photos simply have no
   row). Previously `images` was read in Step 1 only to compute
   image_url and then discarded -- OFF's own image tags (images[].key)
   are not fully reliable (spot-checked cases of front-tagged photos
   that were actually nutrition-panel photos), so OCR needs access to
   every photo per product, not just the one image_url already points
   to. Columns: front_url / nutrition_url / ingredients_url /
   packaging_url (first photo OFF tagged with that category, original
   resolution only -- no .400 thumbnail column, nutrition-panel text is
   often too small for that to be safe for OCR), other_image_urls (JSON
   array of any extra photos beyond the first pick per category -- some
   products have 2-3 same-category photos in different languages, e.g.
   nutrition_en + nutrition_fr, and this is where those live so nothing
   is silently dropped), n_images (total photo count), ocr_status
   ('pending' / 'done_filled' / 'done_no_data', for the OCR pass to
   update as it works through products).

Built in a local temp dir and copied into place -- SQLite over this
mounted path raises 'disk I/O error' on live connections; plain file
copy is unaffected.
"""
import json, sqlite3, re, shutil, tempfile
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

BASE_DIR = Path.cwd()
IN_FILE = BASE_DIR / "build" / "baby_foods_raw.jsonl"
OUT_DB = BASE_DIR / "products.db"

NUTRIENT_MAP = {
    "energy-kcal": "energy_kcal_100g",
    "energy-kj": "energy_kj_100g",
    "proteins": "protein_100g",
    "fat": "fat_100g",
    "saturated-fat": "saturated_fat_100g",
    "carbohydrates": "carbohydrates_100g",
    "sugars": "sugars_100g",
    "fiber": "fiber_100g",
    "sodium": "sodium_100g",
    "salt": "salt_100g",
}

CORE_NUTRIENTS = [
    "energy_kcal_100g", "protein_100g", "fat_100g", "saturated_fat_100g",
    "carbohydrates_100g", "sugars_100g", "fiber_100g", "sodium_mg_100g",
]

IMAGE_CATEGORIES = ("front", "nutrition", "ingredients", "packaging")

def pick_text(lang_list):
    if not lang_list:
        return None
    by_lang = {e.get("lang"): e.get("text") for e in lang_list if e.get("text")}
    for pref in ("en", "main"):
        if by_lang.get(pref):
            return by_lang[pref]
    for e in lang_list:
        if e.get("text"):
            return e["text"]
    return None

def clean_str(v):
    if v is None:
        return None
    v = str(v).strip()
    return v if v else None

def clean_list(v):
    return v if v else None

_HTML_TAG_RE = re.compile(r"<[^>]+>")

def strip_html(text):
    if not text:
        return None
    text = _HTML_TAG_RE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None

def parse_quantity(v):
    if v is None:
        return None
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return f if f != 0 else None

def iso_date(epoch_seconds):
    if epoch_seconds is None:
        return None
    try:
        return datetime.fromtimestamp(int(epoch_seconds), tz=timezone.utc).date().isoformat()
    except (TypeError, ValueError, OSError):
        return None

def image_url(code, images_list, lang):
    """Backward-compatible single front-photo URL (existing frontend field)."""
    if not images_list or not code:
        return None
    candidates = [f"front_{lang}"] if lang else []
    candidates.append("front_en")
    front = None
    for key in candidates:
        for im in images_list:
            if (im.get("key") or "") == key:
                front = im
                break
        if front:
            break
    if front is None:
        for im in images_list:
            if (im.get("key") or "").startswith("front"):
                front = im
                break
    if front is None:
        return None
    imgid = front.get("imgid")
    if imgid is None:
        return None
    digits = re.sub(r"\D", "", str(code))
    if not digits:
        return None
    if len(digits) > 8:
        parts = [p for p in (digits[0:3], digits[3:6], digits[6:9], digits[9:]) if p]
        path = "/".join(parts)
    else:
        path = digits
    return f"https://images.openfoodfacts.org/images/products/{path}/{imgid}.400.jpg"

def img_full_url(code, imgid):
    """Original-resolution image URL (no size suffix)."""
    digits = re.sub(r"\D", "", str(code))
    if not digits:
        return None
    if len(digits) > 8:
        parts = [p for p in (digits[0:3], digits[3:6], digits[6:9], digits[9:]) if p]
        path = "/".join(parts)
    else:
        path = digits
    return f"https://images.openfoodfacts.org/images/products/{path}/{imgid}.jpg"

def image_category(key):
    if not key:
        return "other"
    for c in IMAGE_CATEGORIES:
        if key.startswith(c):
            return c
    return "other"

def pivot_images(code, images_list):
    """One row for this product's product_images entry, or None if it has
    no photos at all. First photo per category goes in its own column;
    everything else (extra same-category photos, untagged photos) goes
    into other_image_urls so nothing is silently dropped."""
    if not images_list:
        return None
    slots = {c: None for c in IMAGE_CATEGORIES}
    leftovers = []
    n = 0
    for im in images_list:
        imgid = im.get("imgid")
        if imgid is None:
            continue
        n += 1
        key = im.get("key")
        url = img_full_url(code, imgid)
        cat = image_category(key)
        if cat in slots and slots[cat] is None:
            slots[cat] = url
        else:
            leftovers.append({"imgid": imgid, "image_key": key, "url": url})
    if n == 0:
        return None
    return {
        "code": code,
        "front_url": slots["front"],
        "nutrition_url": slots["nutrition"],
        "ingredients_url": slots["ingredients"],
        "packaging_url": slots["packaging"],
        "other_image_urls": json.dumps(leftovers, ensure_ascii=False) if leftovers else None,
        "n_images": n,
        "ocr_status": "pending",
    }

def main():
    rows = []
    image_rows = []
    skipped = 0
    n_obsolete = 0
    n_name_fallback = 0
    n_no_name_no_brand = 0
    with open(IN_FILE, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                skipped += 1
                continue

            if rec.get("obsolete"):
                n_obsolete += 1
                continue

            code = rec.get("code")
            lang = clean_str(rec.get("lang"))

            name = pick_text(rec.get("product_name"))
            if not name:
                name = pick_text(rec.get("generic_name"))
                if name:
                    n_name_fallback += 1

            brand = clean_str(rec.get("brands"))
            if not name and not brand:
                n_no_name_no_brand += 1
                continue

            nutrients = {}
            for n in (rec.get("nutriments") or []):
                nm = n.get("name")
                if nm in NUTRIENT_MAP:
                    nutrients[NUTRIENT_MAP[nm]] = n.get("100g")

            sodium_g = nutrients.get("sodium_100g")
            salt_g = nutrients.get("salt_100g")
            if sodium_g is not None:
                sodium_mg = round(sodium_g * 1000, 4)
            elif salt_g is not None:
                sodium_mg = round((salt_g / 2.5) * 1000, 4)
            else:
                sodium_mg = None

            def tags(field):
                v = clean_list(rec.get(field))
                return json.dumps(v, ensure_ascii=False) if v is not None else None

            product_quantity = parse_quantity(rec.get("product_quantity"))
            product_quantity_unit = clean_str(rec.get("product_quantity_unit")) if product_quantity is not None else None

            row = {
                "code": code,
                "name": name,
                "brand": brand,
                "quantity": clean_str(rec.get("quantity")),
                "product_quantity": product_quantity,
                "product_quantity_unit": product_quantity_unit,
                "serving_size": clean_str(rec.get("serving_size")),
                "serving_quantity": clean_str(rec.get("serving_quantity")),
                "categories_tags": tags("categories_tags"),
                "countries_tags": tags("countries_tags"),
                "brand_tags": tags("brands_tags"),
                "image_url": image_url(code, rec.get("images"), lang),
                "lang": lang,
                "last_modified_t": rec.get("last_modified_t"),
                "last_modified_date": iso_date(rec.get("last_modified_t")),
                "energy_kcal_100g": nutrients.get("energy_kcal_100g"),
                "energy_kj_100g": nutrients.get("energy_kj_100g"),
                "protein_100g": nutrients.get("protein_100g"),
                "fat_100g": nutrients.get("fat_100g"),
                "saturated_fat_100g": nutrients.get("saturated_fat_100g"),
                "carbohydrates_100g": nutrients.get("carbohydrates_100g"),
                "sugars_100g": nutrients.get("sugars_100g"),
                "fiber_100g": nutrients.get("fiber_100g"),
                "sodium_mg_100g": sodium_mg,
                "sodium_source": "sodium" if sodium_g is not None else ("salt_derived" if salt_g is not None else None),
                "nutrition_data_per": clean_str(rec.get("nutrition_data_per")),
                "ingredients_text": strip_html(pick_text(rec.get("ingredients_text"))),
                "ingredients_tags": tags("ingredients_tags"),
                "additives_tags": tags("additives_tags"),
                "allergens_tags": tags("allergens_tags"),
                "labels_tags": tags("labels_tags"),
                "food_groups_tags": tags("food_groups_tags"),
                "minerals_tags": tags("minerals_tags"),
                "vitamins_tags": tags("vitamins_tags"),
                "nutriscore_grade": clean_str(rec.get("nutriscore_grade")),
                "nova_group": rec.get("nova_group"),
            }

            for nut in CORE_NUTRIENTS:
                has_val = row.get(nut) is not None
                if nut == "sodium_mg_100g":
                    src = row["sodium_source"]
                else:
                    src = "off_measured" if has_val else None
                row[f"{nut}_source"] = src
                row[f"{nut}_quality_flag"] = "confirmed" if has_val else None

            row["missing_nutrient_count"] = sum(1 for nut in CORE_NUTRIENTS if row.get(nut) is None)

            rows.append(row)

            img_row = pivot_images(code, rec.get("images"))
            if img_row is not None:
                image_rows.append(img_row)

    print(f"Parsed {len(rows)} records ({skipped} bad JSON skipped, {n_obsolete} obsolete excluded, "
          f"{n_no_name_no_brand} dropped for no name+no brand, {n_name_fallback} names filled from generic_name).")

    local_tmp_db = Path(tempfile.mkdtemp()) / "products.db"
    conn = sqlite3.connect(local_tmp_db)
    cur = conn.cursor()

    nutrient_cols_sql = ",\n        ".join(
        f"{nut}_source TEXT,\n        {nut}_quality_flag TEXT" for nut in CORE_NUTRIENTS
    )

    cur.execute(f"""
    CREATE TABLE products (
        code TEXT PRIMARY KEY,
        name TEXT,
        brand TEXT,
        quantity TEXT,
        product_quantity REAL,
        product_quantity_unit TEXT,
        serving_size TEXT,
        serving_quantity TEXT,
        categories_tags TEXT,
        countries_tags TEXT,
        brand_tags TEXT,
        image_url TEXT,
        lang TEXT,
        last_modified_t INTEGER,
        last_modified_date TEXT,
        energy_kcal_100g REAL,
        energy_kj_100g REAL,
        protein_100g REAL,
        fat_100g REAL,
        saturated_fat_100g REAL,
        carbohydrates_100g REAL,
        sugars_100g REAL,
        fiber_100g REAL,
        sodium_mg_100g REAL,
        sodium_source TEXT,
        nutrition_data_per TEXT,
        ingredients_text TEXT,
        ingredients_tags TEXT,
        additives_tags TEXT,
        allergens_tags TEXT,
        labels_tags TEXT,
        food_groups_tags TEXT,
        minerals_tags TEXT,
        vitamins_tags TEXT,
        nutriscore_grade TEXT,
        nova_group INTEGER,
        {nutrient_cols_sql},
        missing_nutrient_count INTEGER
    )
    """)

    cur.execute("""
    CREATE TABLE product_images (
        code TEXT PRIMARY KEY,
        front_url TEXT,
        nutrition_url TEXT,
        ingredients_url TEXT,
        packaging_url TEXT,
        other_image_urls TEXT,
        n_images INTEGER,
        ocr_status TEXT
    )
    """)

    cols = list(rows[0].keys())
    cur.executemany(
        f"INSERT OR IGNORE INTO products ({','.join(cols)}) VALUES ({','.join(['?']*len(cols))})",
        [tuple(r[c] for c in cols) for r in rows]
    )

    img_cols = list(image_rows[0].keys()) if image_rows else []
    if image_rows:
        cur.executemany(
            f"INSERT OR IGNORE INTO product_images ({','.join(img_cols)}) VALUES ({','.join(['?']*len(img_cols))})",
            [tuple(r[c] for c in img_cols) for r in image_rows]
        )

    conn.commit()

    n = cur.execute("SELECT count(*) FROM products").fetchone()[0]
    n_img_products = cur.execute("SELECT count(*) FROM product_images").fetchone()[0]

    dist = Counter(r["missing_nutrient_count"] for r in rows)

    conn.close()
    shutil.copy(local_tmp_db, OUT_DB)

    print(f"\nproducts.db built: {n} rows in products, {len(cols)} columns.")
    print(f"product_images: {n_img_products} products with at least one photo "
          f"({n - n_img_products} products have zero images).")

    print("\nMissing-nutrient-count distribution (0 = all 8 present, 8 = all missing):")
    for k in range(9):
        c = dist.get(k, 0)
        print(f"  {k} missing: {c:5d}  ({round(100*c/n,1)}%)")

    conn2 = sqlite3.connect(OUT_DB)
    cur2 = conn2.cursor()
    checks = ["name", "brand", "image_url", "energy_kcal_100g", "ingredients_text",
              "countries_tags", "last_modified_t", "product_quantity",
              "brand_tags", "minerals_tags"]
    print("\nCoverage checks:")
    for c in checks:
        v = cur2.execute(f"SELECT count(*) FROM products WHERE {c} IS NOT NULL").fetchone()[0]
        print(f"  has {c}: {v} ({round(100*v/n,1)}%)")
    conn2.close()

if __name__ == "__main__":
    main()


## Result

`products.db` -- table `products`, 6,956 rows (barcode `code` primary key), 53 columns; table `product_images`, 6,700 rows (one per product with at least one OFF photo, `code` primary key).

Row count changed from the 2026-08-31 build (6,920 -> 6,956) purely because the 2026-09-24 `food.parquet` pull is a newer OFF snapshot: 38 products newly matched the `en:baby-foods` filter, 2 previously-matching products dropped out (most likely reclassified or newly missing both name and brand) -- same filter and cleaning logic both times, this is upstream data drift, not a rule change.

| column | notes |
|---|---|
| code | barcode, primary key |
| name | falls back to `generic_name` when `product_name` is empty |
| brand | renamed from `brands` (2026-08-31); free text |
| quantity, serving_size | free text as shipped by OFF; kept as a display fallback alongside the parsed columns below |
| product_quantity(+unit), serving_quantity | structured numeric versions of quantity/serving_size; `product_quantity` is nulled (unit nulled with it) when OFF's own structured value parses to exactly 0, since that's OFF's own missing-data artifact, not a real packet weight |
| categories_tags, countries_tags, brand_tags | JSON arrays of OFF tags, `en:`-prefixed where OFF taxonomy applies; NULL (not `'[]'`) when empty. Kept in this raw form in storage -- it's what claim-keyword and category-mapping logic matches against -- with any human-friendly relabeling (`en:organic` -> "Organic") left to the display layer |
| image_url | only set when OFF's `images[].key` explicitly marks a front photo; NULL otherwise. Kept for backward compatibility with existing frontend code -- `product_images` (below) is the complete picture and what OCR should use |
| lang | product's primary language; used by `image_url` to prefer a same-language front photo, not used elsewhere in the build |
| last_modified_t | OFF's last-modified timestamp, Unix epoch (kept for sorting/precision) |
| last_modified_date | `last_modified_t` converted to an ISO 8601 date, so the frontend's "last updated" field doesn't need to convert the epoch itself |
| energy_kcal_100g, energy_kj_100g | both kept (not deduped) -- FSANZ-style labeling conventions expect both units shown, so the choice is left to the frontend/compliance requirements rather than collapsed here |
| protein_100g ... sodium_mg_100g | remaining 6 core nutrients, per 100g -- confirmed all 8 nutrients are read from OFF's `"100g"` sub-field consistently, never `"serving"` |
| {nutrient}_source (all 8) | new 2026-09-24, extends the pre-existing `sodium_source` pattern to every core nutrient. `off_measured` / `salt_derived` (sodium only) when this build populated the value, else NULL. Set to `ocr_extracted` by the OCR pass once a value is read off a photo |
| {nutrient}_quality_flag (all 8) | new 2026-09-24, paired with `_source`. `confirmed` when populated by this build, else NULL. Set to `unverified` by the OCR pass -- OCR'd numbers are not presented as equal to OFF's own data until spot-checked |
| missing_nutrient_count | new 2026-09-24: how many of the 8 core nutrients are NULL for this product (0-8), computed once here instead of via a separate query each time |
| nutrition_data_per | whether source nutrient values were declared per-100g or per-serving |
| ingredients_text | free text; residual HTML markup (e.g. stray `<span class="allergen">` tags) is stripped. Case is left as-is -- lowercasing loses information (proper nouns, "GMO"-style abbreviations); a separate lowercase index field is the better fix if search matching needs it later |
| ingredients_tags, additives_tags, allergens_tags, labels_tags | machine-matchable counterparts to `ingredients_text`, used for allergen/claim/additive matching rather than display |
| food_groups_tags | a coarser OFF taxonomy than `categories_tags` (e.g. 2 tags vs. 8), not a duplicate |
| minerals_tags, vitamins_tags | for future category-mapping/claim work |
| nutriscore_grade | kept in the DB, but flagged not to surface in the frontend for this subset -- the large majority of rows are OFF's own `'not-applicable'` placeholder, so it carries no signal here |
| nova_group | real distribution across this subset (1/2/3/4/missing all represented), kept |

`product_images` (new table, 2026-09-24) -- one row per product that has at least one OFF photo (products with zero photos simply have no row here):

| column | notes |
|---|---|
| code | join key back to `products`, primary key |
| front_url, nutrition_url, ingredients_url, packaging_url | first photo OFF tagged with that category (matched on `images[].key` prefix), original resolution only -- no `.400` thumbnail column, nutrition-panel text is often too small for that to be safe for OCR |
| other_image_urls | JSON array of any photos beyond the first pick per category (e.g. a product with both `nutrition_en` and `nutrition_fr` photos), plus untagged photos. NULL when there's nothing left over -- nothing is silently dropped in favor of the four category columns |
| n_images | total photo count for that product across all categories |
| ocr_status | `pending` / `done_filled` / `done_no_data`, updated by the OCR pass as it works through candidates |

Missing-nutrient-count distribution (2026-09-24 build, 6,956 products): 3,303 (47.5%) have all 8 nutrients; 1,450 (20.8%) are missing all 8; the remaining ~31.7% are partially missing. Of the 3,653 products missing at least one nutrient, 3,463 have at least one photo to try OCR against and 190 have none (unrecoverable by OCR). Full breakdown and the OCR workflow itself are in `OCR_HANDOFF.md`, not duplicated here since that file is what the OCR pass actually reads.

Removed this pass (2026-08-31, still true 2026-09-24), all previously present:
- Rows with both `name` and `brand` NULL -- unfindable, unrecognizable products (116 on 2026-08-31, 118 on 2026-09-24's snapshot).
- **`manufacturing_places_tags`** (10.1% coverage, no backfill source -- and not something OCR can fill either, since it isn't printed on a nutrition panel).
- **`link`** (10.3% coverage, low value).
- **`nucleotides_tags`** (2.0% coverage -- too sparse to justify keeping as a forward-looking placeholder).

Not persisted as columns on `products` (used only during the build, then discarded): `images` raw metadata is no longer fully discarded -- it now also populates `product_images` -- but is not itself stored as a column; `obsolete` (used to exclude discontinued rows); `generic_name` (only used as a fallback into `name`).

Deferred, not part of this rebuild: adding OFF's `traces_tags` (possible/trace allergens, distinct from confirmed `allergens_tags`) if allergen safety messaging is built later.